In [1]:
import os
from abc import ABC, abstractmethod
from enum import Enum
from typing import List

import anthropic
from anthropic.types import UserMessageParam, AssistantMessageParam
from openai import OpenAI
from openai.types.chat import ChatCompletionSystemMessageParam, ChatCompletionUserMessageParam, ChatCompletionAssistantMessageParam

In [2]:
class MsgType(Enum):
    SYSTEM = "System"
    USER = "User"
    ASSISTANT = "Assistant"

In [3]:
class Message(ABC):

    def __init__(self, msg_type: MsgType):
        self.msg_type = msg_type

    @abstractmethod
    def summary(self) -> str:
        pass

    def __repr__(self) -> str:
        return f"Message({self.msg_type.value}): {self.summary()}"

In [4]:
class SystemMessage(Message):

    def __init__(self, description: str):
        super().__init__(MsgType.SYSTEM)
        self.description = description

    def summary(self) -> str:
        return self.description

In [5]:
class UserMessage(Message):

    def __init__(self, request: str):
        super().__init__(MsgType.USER)
        self.request = request

    def summary(self) -> str:
        return self.request

In [6]:
class AssistantMessage(Message):

    def __init__(self, content: str):
        super().__init__(MsgType.ASSISTANT)
        self.content = content

    def summary(self) -> str:
        return self.content

In [7]:
system_message = SystemMessage("You're a learning assistant for the book 《从大模型到智能体》 (From LLM to Agent). "
                               "You help beginners learn how to build AI agents.")

In [8]:
class Memory:

    def __init__(self):
        self.messages: List[Message] = []

    def add(self, message: Message):
        self.messages.append(message)

    def reset(self):
        self.messages = []

    def history(self) -> str:
        return "\n".join(
            f"[{msg.msg_type.value}]: {msg.summary()}" for msg in self.messages
        )

In [9]:
class LLMProvider(ABC):

    @abstractmethod
    def call(self, system: SystemMessage, messages: List[Message], model=None, max_tokens=4096):
        pass

In [10]:
class OpenAIProvider(LLMProvider):

    def __init__(self):
        self.client = OpenAI(
            api_key=os.getenv("OPENAI_API_KEY"),
            base_url=None
        )
        self.default_model = "gpt-5.4-mini"

    def call(self, system: SystemMessage, messages: List[Message], model=None, max_tokens=4096):
        if system:
            messages = [system] + messages
        kwargs = dict(
            model=model or self.default_model,
            max_completion_tokens=max_tokens,
            messages=self._message_schema(messages)
        )
        response = self.client.chat.completions.create(**kwargs)
        return self._response_schema(response)

    @staticmethod
    def _message_schema(messages: List[Message]):
        schema = []
        for msg in messages:
            if isinstance(msg, SystemMessage):
                schema.append(ChatCompletionSystemMessageParam(role="system", content=msg.description))
            elif isinstance(msg, UserMessage):
                schema.append(ChatCompletionUserMessageParam(role="user", content=msg.request))
            elif isinstance(msg, AssistantMessage):
                schema.append(ChatCompletionAssistantMessageParam(role="assistant", content=msg.content))
        return schema

    @staticmethod
    def _response_schema(response):
        if response is None:
            return AssistantMessage("")

        message = response.choices[0].message
        return AssistantMessage(message.content or "")

In [11]:
class ClaudeProvider(LLMProvider):

    def __init__(self):
        self.client = anthropic.Anthropic(
            api_key=os.getenv("ANTHROPIC_API_KEY")
        )
        self.default_model = "claude-haiku-4-5"

    def call(self, system: SystemMessage, messages: List[Message], model=None, max_tokens=4096):
        kwargs = dict(
            model=model or self.default_model,
            max_tokens=max_tokens,
            messages=self._message_schema(messages)
        )
        if system:
            kwargs["system"] = system.summary()
        response = self.client.messages.create(**kwargs)
        return self._response_schema(response)

    @staticmethod
    def _message_schema(messages: List[Message]):
        schema = []
        for msg in messages:
            if isinstance(msg, UserMessage):
                schema.append(UserMessageParam(role="user", content=msg.request))
            elif isinstance(msg, AssistantMessage):
                schema.append(AssistantMessageParam(role="assistant", content=msg.content))
        return schema

    @staticmethod
    def _response_schema(response):
        if response is None:
            return AssistantMessage("")

        text = "".join(block.text for block in response.content if block.type == "text")
        return AssistantMessage(text)

In [12]:
class Agent:

    def __init__(self, provider: LLMProvider, system: SystemMessage):
        self.provider = provider
        self.system = system
        self.memory = Memory()

    def chat(self, questions: List[str] = None, system: SystemMessage = None):
        questions = iter(questions) if questions is not None else None  # preset questions, or None to type
        while True:
            message = next(questions, "/quit") if questions else input("YOU: ")
            print(f"\nYOU: {message}")
            if message.strip().lower() == "/quit":
                break
            if not message.strip():
                continue

            user_message = UserMessage(message)
            response = self.provider.call(system or self.system, self.memory.messages + [user_message])
            self.memory.add(user_message)
            self.memory.add(response)
            print(f"AI: {response.summary()}")


In [ ]:
provider = OpenAIProvider()
agent = Agent(provider, system_message)
agent.chat(["你好，我叫小明，正在学这本书。", "我叫什么名字？"])  # with memory
# or call agent.chat() without questions to type your own